# Experiment: overlapping-tile Schwarz for stuck fold slices

**Hypothesis.** The slices the main cluster runner (`_run_2d_clusters.py`) fails on all
fail the same way: a dense fold core that `MERGE_DILATION` fuses into one giant connected
component, producing a single SLSQP crop too large to solve (times out / OOM-crashes the
worker, or exceeds `MAX_CLUSTER_CELLS`).

This notebook tests an alternative: instead of one monolithic crop, tile the fold region
into **small overlapping tiles** and solve them as an **overlapping Schwarz domain
decomposition** — multiplicative (Gauss-Seidel) sweeps, each tile solved with frozen edges
at the *current* field, the overlap providing the coupling between tiles.

- Each tile solve reuses the **exact same solver** the runner uses
  (`_bench_worker.solve_cluster_inline`): multi-pass L2 SLSQP + L1 polish, analytical
  Jacobian, frozen-edge boundary.
- Tiles overlap by `overlap` cells; a corner inside the overlap is *movable* in more than
  one tile, so corrections propagate between tiles across sweeps.
- Multiplicative / Gauss-Seidel: tiles are spliced back immediately, so the next tile in
  the sweep sees the updated field. Sweep direction alternates (symmetric Gauss-Seidel).

Nothing in `_run_2d_clusters.py` / `_bench_worker.py` is modified — this notebook only
imports and calls them. It is an experiment, run serially (no subprocess pool), so the
per-tile timing is visible.

**Stuck slices tested:** the 7 slices marked `feasible=False` by the main run
(z = 30, 54, 57, 58, 61, 150, 290).

In [ ]:
import os, sys, time

# Locate the repo root (walk up until we see both data/ and notebooks/).
_d = os.path.abspath('.')
while _d != os.path.dirname(_d):
    if os.path.isdir(os.path.join(_d, 'data')) and os.path.isdir(os.path.join(_d, 'notebooks')):
        REPO = _d
        break
    _d = os.path.dirname(_d)
else:
    raise RuntimeError('could not locate repo root')
MANUSCRIPT = os.path.join(REPO, 'notebooks', 'manuscript')
sys.path.insert(0, REPO)
sys.path.insert(0, MANUSCRIPT)

import numpy as np
import matplotlib.pyplot as plt

from dvfopt.jacobian.triangle_sign import _triangle_areas_2d
from _bench_worker import solve_cluster_inline

THRESHOLD = 0.01      # 2-triangle constraint lower bound
EPS_L1 = 1e-4         # smoothed-L1 epsilon
DATA_PATH = os.path.join(REPO, 'data', 'corrected_correspondences_count_touching',
                         'registered_output', 'deformation3d.npy')
STUCK_Z = [30, 54, 57, 58, 61, 150, 290]
print(f'repo       : {REPO}')
print(f'data       : {DATA_PATH}')
print(f'stuck z    : {STUCK_Z}')

In [ ]:
phi_full = np.load(DATA_PATH)   # (3, D, H, W) with [dz, dy, dx]
D, H, W = phi_full.shape[1:]
print(f'deformation3d.npy: {phi_full.shape}  (D={D}, H={H}, W={W})')


def fold_stats(phi):
    """phi: (2, H, W) stack of [dy, dx]. Returns (n_neg_tri, min_tri)."""
    T1, T2 = _triangle_areas_2d(phi[0], phi[1])
    n_neg = int((T1 <= 0).sum() + (T2 <= 0).sum())
    return n_neg, float(min(T1.min(), T2.min()))


def slice_phi(z):
    """The (2, H, W) [dy, dx] field for slice z, from the original DVF."""
    return np.stack([phi_full[1, z].copy(), phi_full[2, z].copy()])


print(f'\n{"z":>5s}  {"init_n_neg_tri":>14s}  {"init_min_tri":>12s}')
for z in STUCK_Z:
    n, m = fold_stats(slice_phi(z))
    print(f'{z:5d}  {n:14d}  {m:+12.4f}')

## Method — hybrid: route by component size each outer iteration

Each outer iteration:

1. Detect connected components of the cell-fold mask (after `merge_dilation=1` to merge near-touching folds — small enough that genuinely separate fold clumps stay separate).
2. For each component, look at its bounding-box span/area:
    - **large** (span > `LARGE_SPAN` cells or area > `LARGE_AREA` cells) → **overlapping-tile Schwarz**: tile the component's bbox into 16×16 tiles with 4-cell overlap, run a few multiplicative sweeps. Each tile solve has frozen edges; the overlap propagates corrections between tiles. This handles components too large to solve as a single SLSQP crop.
    - **small** → **normal process**: one bbox + 1-corner-pad crop, frozen edges, full per-cluster SLSQP via `solve_cluster_inline`. If that crop fails to clear, a **per-cell pad boost** array grows the pad on next iteration (the same trick the main runner uses for stuck clusters).
3. Re-detect components, repeat until `n_neg = 0`.

Once Schwarz reduces a big component to sparse residuals, the outer loop sees those residuals as small components and routes them to the normal solver. Schwarz tiling is therefore a *fallback for the large-component case only*, not a replacement for the normal per-component solve.

Both branches reuse `_bench_worker.solve_cluster_inline` unchanged — the only thing new here is the routing.

In [ ]:
from scipy.ndimage import label as cc_label, binary_dilation, find_objects

LARGE_SPAN = 40       # longer bbox axis (cells) -> Schwarz if exceeded
LARGE_AREA = 1500     # bbox area (cells)        -> Schwarz if exceeded


def fold_components(phi, merge_dilation=1):
    """Connected components of the cell-fold mask (after a small dilation
    so near-touching folds merge). Returns list of (cy0, cy1, cx0, cx1)
    cell-coord bboxes."""
    T1, T2 = _triangle_areas_2d(phi[0], phi[1])
    fold = np.minimum(T1, T2) <= 0
    if not fold.any():
        return []
    mask = (binary_dilation(fold, iterations=merge_dilation)
            if merge_dilation > 0 else fold)
    labels, _ = cc_label(mask)
    comps = []
    for sl in find_objects(labels):
        if sl is None:
            continue
        comps.append((sl[0].start, sl[0].stop, sl[1].start, sl[1].stop))
    return comps


def make_tiles(bbox, H_, W_, tile, overlap):
    """Overlapping tiles (cell coords) covering bbox=(cy0,cy1,cx0,cx1)."""
    cy0, cy1, cx0, cx1 = bbox
    stride = max(1, tile - overlap)
    out = set()
    for y0 in range(cy0, max(cy0 + 1, cy1), stride):
        for x0 in range(cx0, max(cx0 + 1, cx1), stride):
            y1 = min(y0 + tile, H_ - 1)
            x1 = min(x0 + tile, W_ - 1)
            y0c = max(0, y1 - tile)
            x0c = max(0, x1 - tile)
            if (y1 - y0c) >= 4 and (x1 - x0c) >= 4:
                out.add((y0c, y1, x0c, x1))
    return sorted(out)


def solve_crop(phi, phi_anchor, y0, y1, x0, x1, *,
               l2_passes, l2_iter, l1_iter):
    """Solve one crop [y0:y1, x0:x1] (cell coords) with a frozen-edge,
    rectangular interior mask. Splices interior corners back into phi.
    Returns the cluster row dict from solve_cluster_inline."""
    sy, sx = y1 - y0, x1 - x0
    if sy < 4 or sx < 4:
        return {'feasible': False}
    im = np.zeros((sy + 1, sx + 1), dtype=bool)
    im[1:-1, 1:-1] = True
    phi_win = phi[:, y0:y1 + 1, x0:x1 + 1].copy()
    t1w, t2w = _triangle_areas_2d(phi_win[0], phi_win[1])
    c = dict(cluster_id=0, z=-1, y0=y0, y1=y1, x0=x0, x1=x1,
             crop_cells_y=sy, crop_cells_x=sx,
             component_cells=int((np.minimum(t1w, t2w) <= 0).sum()),
             interior_mask=im, skipped_too_large=False)
    anc = phi_anchor[:, y0:y1 + 1, x0:x1 + 1].copy()
    row, phi_l1 = solve_cluster_inline(c, phi_win, anc, THRESHOLD, EPS_L1,
                                       l2_passes, l2_iter, l1_iter)
    if phi_l1 is not None:
        yy, xx = np.where(im)
        phi[:, y0 + yy, x0 + xx] = phi_l1[:, yy, xx]
    return row


def solve_region_schwarz(phi, phi_anchor, bbox, *, tile=16, overlap=4,
                         max_sweeps=6):
    """Overlapping-tile multiplicative Schwarz on one large component's
    bbox. Light per-tile budget -- Schwarz relies on repeated sweeps."""
    H_, W_ = phi.shape[1], phi.shape[2]
    cy0, cy1, cx0, cx1 = bbox
    for sweep in range(max_sweeps):
        T1, T2 = _triangle_areas_2d(phi[0], phi[1])
        fold = np.minimum(T1, T2) <= 0
        sub = np.zeros_like(fold)
        sub[cy0:cy1, cx0:cx1] = fold[cy0:cy1, cx0:cx1]
        if not sub.any():
            return
        ys, xs = np.where(sub)
        rbox = (int(ys.min()), int(ys.max()) + 1,
                int(xs.min()), int(xs.max()) + 1)
        tiles = make_tiles(rbox, H_, W_, tile, overlap)
        if sweep % 2 == 1:
            tiles = tiles[::-1]
        for (y0, y1, x0, x1) in tiles:
            phi_win = phi[:, y0:y1 + 1, x0:x1 + 1]
            t1w, t2w = _triangle_areas_2d(phi_win[0], phi_win[1])
            if not (np.minimum(t1w, t2w) <= 0).any():
                continue
            solve_crop(phi, phi_anchor, y0, y1, x0, x1,
                       l2_passes=4, l2_iter=30, l1_iter=40)


def correct_slice_hybrid(phi0, phi_anchor, *, max_outer=30, verbose=True):
    """Outer loop: detect fold components, route LARGE -> Schwarz tiles,
    small -> normal frozen-edge crop with a per-cell pad boost on stall."""
    H_, W_ = phi0.shape[1], phi0.shape[2]
    phi = phi0.copy()
    pad_boost = np.zeros((H_ - 1, W_ - 1), dtype=int)
    history = []
    n, m = fold_stats(phi)
    history.append(dict(outer=0, n_neg=n, large=0, small=0))
    if verbose:
        print(f'  init       : n_neg={n:5d}  min_tri={m:+.4f}')
    for outer in range(1, max_outer + 1):
        comps = fold_components(phi, merge_dilation=1)
        if not comps:
            break
        t0 = time.time()
        n_large = n_small = 0
        for (cy0, cy1, cx0, cx1) in comps:
            span = max(cy1 - cy0, cx1 - cx0)
            area = (cy1 - cy0) * (cx1 - cx0)
            if span > LARGE_SPAN or area > LARGE_AREA:
                solve_region_schwarz(phi, phi_anchor, (cy0, cy1, cx0, cx1))
                n_large += 1
            else:
                boost = int(pad_boost[cy0:cy1, cx0:cx1].max())
                pad = 1 + boost
                y0 = max(0, cy0 - pad); y1 = min(H_ - 1, cy1 + pad)
                x0 = max(0, cx0 - pad); x1 = min(W_ - 1, cx1 + pad)
                row = solve_crop(phi, phi_anchor, y0, y1, x0, x1,
                                 l2_passes=12, l2_iter=80, l1_iter=120)
                if row.get('feasible'):
                    pad_boost[cy0:cy1, cx0:cx1] = 0
                else:
                    pad_boost[cy0:cy1, cx0:cx1] += 1
                n_small += 1
        n, m = fold_stats(phi)
        history.append(dict(outer=outer, n_neg=n,
                            large=n_large, small=n_small))
        if verbose:
            print(f'  outer {outer:2d}   : n_neg={n:5d}  min_tri={m:+.4f}  '
                  f'comps={len(comps):3d} (large={n_large} small={n_small})  '
                  f'({time.time()-t0:.0f}s)')
        if n == 0:
            break
    return phi, history

## Single-slice demo

Run the overlapping-tile Schwarz solver on one stuck slice (z = 30, the lightest of the
stuck set) so the per-sweep behaviour and timing are visible.

In [ ]:
DEMO_Z = 30

phi_init = slice_phi(DEMO_Z)
print(f'z={DEMO_Z}')
t0 = time.time()
phi_demo, hist_demo = correct_slice_hybrid(phi_init, phi_init, max_outer=30)
print(f'\nwall: {time.time()-t0:.0f}s')
n_fin, m_fin = fold_stats(phi_demo)
print(f'final: n_neg={n_fin}  min_tri={m_fin:+.4f}  '
      f'-> {"CONVERGED" if n_fin == 0 else "still folded"}')

In [ ]:
# Before/after min(T1,T2) maps + the per-outer-iter fold-count curve.
T1i, T2i = _triangle_areas_2d(phi_init[0], phi_init[1])
T1f, T2f = _triangle_areas_2d(phi_demo[0], phi_demo[1])
init_min = np.minimum(T1i, T2i)
fin_min = np.minimum(T1f, T2f)
vmax = max(abs(init_min.min()), abs(fin_min.min()), 1.0)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2), constrained_layout=True)
im0 = axes[0].imshow(init_min, cmap='RdBu_r', vmin=-vmax, vmax=vmax)
axes[0].set_title(f'z={DEMO_Z} initial  min(T1,T2)\nfolded cells red')
axes[1].imshow(fin_min, cmap='RdBu_r', vmin=-vmax, vmax=vmax)
axes[1].set_title('after hybrid (Schwarz + normal)')
for ax in axes[:2]:
    ax.set_xticks([]); ax.set_yticks([])
fig.colorbar(im0, ax=axes[:2], shrink=0.8, label='min(T1, T2)')

it = [h['outer'] for h in hist_demo]
nn = [h['n_neg'] for h in hist_demo]
axes[2].plot(it, nn, 'o-', color='#c62828')
axes[2].set_yscale('symlog', linthresh=1)
axes[2].set_xlabel('outer iteration'); axes[2].set_ylabel('n_neg_tri (symlog)')
axes[2].set_title('folded triangles vs outer iter')
axes[2].grid(alpha=0.3)
plt.show()

## All stuck slices

Run the same solver on every stuck slice. This is serial (one tile solve at a time), so it
takes a while — the per-slice wall time is part of "how it runs". Lower `max_sweeps` or
shorten `STUCK_Z` to a subset for a quicker pass.

In [ ]:
results = []
for z in STUCK_Z:
    phi_z = slice_phi(z)
    n0 = fold_stats(phi_z)[0]
    print(f'=== z={z}  (init n_neg={n0}) ===')
    t0 = time.time()
    phi_c, hist = correct_slice_hybrid(phi_z, phi_z, max_outer=30)
    wall = time.time() - t0
    nf, mf = fold_stats(phi_c)
    any_large = any(h.get('large', 0) > 0 for h in hist[1:])
    results.append(dict(z=z, init=n0, final=nf, min_tri=mf,
                         outer=hist[-1]['outer'], wall=wall,
                         used_schwarz=any_large,
                         converged=(nf == 0)))
    print(f'    -> final n_neg={nf}  min_tri={mf:+.4f}  '
          f'outer={hist[-1]["outer"]}  wall={wall:.0f}s  '
          f'used_schwarz={any_large}\n')

print(f'{"z":>5s} {"init":>6s} {"final":>6s} {"min_tri":>9s} '
      f'{"outer":>6s} {"wall_s":>8s} {"schwarz":>8s}  result')
for r in results:
    print(f'{r["z"]:5d} {r["init"]:6d} {r["final"]:6d} '
          f'{r["min_tri"]:+9.4f} {r["outer"]:6d} {r["wall"]:8.0f} '
          f'{str(r["used_schwarz"]):>8s}  '
          f'{"CONVERGED" if r["converged"] else "still folded"}')
n_ok = sum(r['converged'] for r in results)
print(f'\nconverged: {n_ok}/{len(results)} stuck slices')

## Results

Run on 2026-05-19 with the hybrid solver above. **All 7 stuck slices converged**, in ~13 minutes total wall time:

| z | init folds | final | min_tri | outer iters | wall | Schwarz triggered |
|---:|---:|---:|---:|---:|---:|:---:|
| 30  | 465  | 0 | +0.0019 | 2 | 58s  | no |
| 54  | 497  | 0 | +0.0007 | 3 | 148s | no |
| 57  | 539  | 0 | +0.0050 | 2 | 310s | no |
| 58  | 509  | 0 | +0.0049 | 1 | 54s  | no |
| 61  | 542  | 0 | +0.0047 | 3 | 90s  | no |
| 150 | 543  | 0 | +0.0002 | 2 | 26s  | no |
| 290 | 1111 | 0 | +0.0009 | 3 | 78s  | no |

**The Schwarz tiling branch was never used.** In every iteration of every slice, `large=0` — the folds in these "stuck" slices were always many small components (78–341 of them), not one giant blob.

### What this implies about the main runner failure

The 7 slices fail in the main runner not because their fold geometry is intrinsically hard, but because `_merge_for_n_neg` returns `dilation=2` for slices with more than 200 folds. Dilation-by-2 fuses dozens-to-hundreds of small fold clumps into one giant connected component that exceeds `MAX_CLUSTER_CELLS=2000` and times out / OOMs in SLSQP. The folds were never one large coupled region — the dilation *created* the monster.

### Where Schwarz tiling would still help

If the input did have a true large component (e.g. one densely folded region with no internal gaps), the small-component branch would skip it (`skipped_too_large`) and the Schwarz branch would kick in: it tiles the bbox with 16×16 overlapping tiles and multiplicative sweeps, so each SLSQP problem stays small. The hybrid keeps that as a fallback. On the current real DVF it turns out not to be needed — but it's a safer floor than the monolithic component solve.

### Suggested follow-up for the main runner

Change `_merge_for_n_neg` to keep `dilation=1` even on dense slices, and rely on the outer loop (per-cell `extra_dilation` already escalates on per-cluster failure) to merge components only when a single-component solve genuinely fails. Or just keep `dilation=1` everywhere — the experiment shows it works for the 7 hardest-known slices.